### Magnitudes de incendios

In [1]:
####################################################################
##  First part: import libraries                                  #
##################################################################


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline  
import os
import rasterio
import rasterio.features
import rasterio.warp
from osgeo import gdal
from matplotlib import pyplot
from rasterio.plot import show 
import geopandas as gpd
from osgeo import ogr, osr
import math



c:\Users\giambelluca.144055\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\giambelluca.144055\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
directory_raster=input('Where are raster layers?: ')
# C:\Users\giambelluca.144055\OneDrive\Incendios\CCDC_tiempos\Navarra
os.chdir(directory_raster)

In [4]:
######################################################################
## Second part: upload SHP and resters of CCDC_Raster_Conditioning  #
####################################################################

B_t1=gdal.Open('Navarra_t1_End_Mag_Prob.tif') 
B_t2=gdal.Open('Navarra_t2_End_Mag_Prob.tif')
B_t3=gdal.Open('Navarra_t3_End_Mag_Prob.tif')
B_t4=gdal.Open('Navarra_t4_End_Mag_Prob.tif')
B_t5=gdal.Open('Navarra_t5_End_Mag_Prob.tif')
B_t6=gdal.Open('Navarra_t6_End_Mag_Prob.tif')

#Extract the first band... in this case tBreak with the date of change

B1_t1=B_t1.GetRasterBand(1).ReadAsArray().astype(np.float32)
B1_t2=B_t2.GetRasterBand(1).ReadAsArray().astype(np.float32)
B1_t3=B_t3.GetRasterBand(1).ReadAsArray().astype(np.float32)
B1_t4=B_t4.GetRasterBand(1).ReadAsArray().astype(np.float32)
B1_t5=B_t5.GetRasterBand(1).ReadAsArray().astype(np.float32)
B1_t6=B_t6.GetRasterBand(1).ReadAsArray().astype(np.float32)

#Extract the second band... in this case magnitude with the date of change

B2_t1=B_t1.GetRasterBand(2).ReadAsArray().astype(np.float32)
B2_t2=B_t2.GetRasterBand(2).ReadAsArray().astype(np.float32)
B2_t3=B_t3.GetRasterBand(2).ReadAsArray().astype(np.float32)
B2_t4=B_t4.GetRasterBand(2).ReadAsArray().astype(np.float32)
B2_t5=B_t5.GetRasterBand(2).ReadAsArray().astype(np.float32)
B2_t6=B_t6.GetRasterBand(2).ReadAsArray().astype(np.float32)

# Extract the third band... in this case changeProb with the probability of change
B3_t1=B_t1.GetRasterBand(3).ReadAsArray().astype(np.float32)
B3_t2=B_t2.GetRasterBand(3).ReadAsArray().astype(np.float32)
B3_t3=B_t3.GetRasterBand(3).ReadAsArray().astype(np.float32)
B3_t4=B_t4.GetRasterBand(3).ReadAsArray().astype(np.float32)
B3_t5=B_t5.GetRasterBand(3).ReadAsArray().astype(np.float32)
B3_t6=B_t6.GetRasterBand(3).ReadAsArray().astype(np.float32)

In [5]:
############################################################################################
## Third part: define a function for extract date, magnitude and probability of change    #
##########################################################################################

def parameters(year):
        yafter=year+1
        y=year
        
        # Get change occured in x year
        capa_1 = np.where((B1_t1 < yafter) & (B1_t1 >=y) , B1_t1, 0)
        capa_2 = np.where((B1_t2 < yafter) & (B1_t2 >=y) , B1_t2, 0)
        capa_3 = np.where((B1_t3 < yafter) & (B1_t3 >=y) , B1_t3, 0)
        capa_4 = np.where((B1_t4 < yafter) & (B1_t4 >=y) , B1_t4, 0)
        capa_5 = np.where((B1_t5 < yafter) & (B1_t5 >=y) , B1_t5, 0)
        capa_6 = np.where((B1_t6 < yafter) & (B1_t6 >=y) , B1_t6, 0)

        t=capa_1+capa_2+capa_3+capa_4+capa_5+capa_6

        #Get magnitude of change

        Mag1 = np.where((capa_1 !=0) , B2_t1, 0)
        Mag2 = np.where((capa_2 !=0) , B2_t2, 0)
        Mag3 = np.where((capa_3 !=0) , B2_t3, 0)
        Mag4 = np.where((capa_4 !=0) , B2_t4, 0)
        Mag5 = np.where((capa_5 !=0) , B2_t5, 0)
        Mag6 = np.where((capa_6 !=0) , B2_t6, 0)

        mag=Mag1+Mag2+Mag3+Mag4+Mag5+Mag6

        del Mag1,Mag2,Mag3,Mag4,Mag5,Mag6

        # Obtain the probability of change for events that occurred in x year

        Prob1 = np.where((capa_1 !=0) , B3_t1, 0)
        Prob2 = np.where((capa_2 !=0) , B3_t2, 0)
        Prob3 = np.where((capa_3 !=0) , B3_t3, 0)
        Prob4 = np.where((capa_4 !=0) , B3_t4, 0)
        Prob5 = np.where((capa_5 !=0) , B3_t5, 0)
        Prob6 = np.where((capa_6 !=0) , B3_t6, 0)

        prob=Prob1+Prob2+Prob3+Prob4+Prob5+Prob6

        del capa_1,capa_2,capa_3,capa_4,capa_5,capa_6
        del Prob1,Prob2,Prob3,Prob4,Prob5,Prob6
        
        probab = np.where((prob ==1), prob, 0)
        magnitude = np.where((probab ==1) , mag, 0)
        time=np.where((probab ==1) , t, 0)
        
        return time,magnitude,probab
    

In [6]:
####################################################################
## Third part: get change for each year                           #
##################################################################
t_1985,mag_1985,prob_1985=parameters(1985)
t_1986,mag_1986,prob_1986=parameters(1986)
t_1987,mag_1987,prob_1987=parameters(1987)
t_1988,mag_1988,prob_1988=parameters(1988)
t_1989,mag_1989,prob_1989=parameters(1989)
t_1990,mag_1990,prob_1990=parameters(1990)
t_1991,mag_1991,prob_1991=parameters(1991)
t_1992,mag_1992,prob_1992=parameters(1992)
t_1993,mag_1993,prob_1993=parameters(1993)
t_1994,mag_1994,prob_1994=parameters(1994)
t_1995,mag_1995,prob_1995=parameters(1995)
t_1996,mag_1996,prob_1996=parameters(1996)
t_1997,mag_1997,prob_1997=parameters(1997)
t_1998,mag_1998,prob_1998=parameters(1998)
t_1999,mag_1999,prob_1999=parameters(1999)
t_2000,mag_2000,prob_2000=parameters(2000)
t_2001,mag_2001,prob_2001=parameters(2001)
t_2002,mag_2002,prob_2002=parameters(2002)
t_2003,mag_2003,prob_2003=parameters(2003)
t_2004,mag_2004,prob_2004=parameters(2004)

In [ ]:
mag_1986.shape


In [ ]:
####################################################################
## Fourth part: Statistics                                        #
##################################################################
SHP.plot()

In [ ]:
# with rasterio.open('array.tif', 'w', driver='GTiff', width=mag_1985.shape[1], height=mag_1985.shape[0], count=1, dtype=rasterio.float32, crs='+proj=longlat +datum=WGS84') as dst:
#     dst.write(mag_1985, 1)

# plt.imshow(b, cmap='gray')  # Ajusta la cmap según tus preferencias
# plt.colorbar()
# plt.title('Magnitude 1985')
# plt.show()

In [7]:
import numpy as np
import gdal

def array_to_raster(array_1,array_2,nombre):
    dst_filename = nombre
    x_pixels = array_1.shape[1]  # Obtener el número de columnas del array
    y_pixels = array_1.shape[0]  # Obtener el número de filas del array
    PIXEL_SIZE = 0.000269494585235856472
#     PIXEL_SIZE =30
    x_min = -3.0000137228455541
    y_max = 44.0001114368730484
    wkt_projection = 'EPSG:4326'

    driver = gdal.GetDriverByName('GTiff')
    dataset = driver.Create(dst_filename, x_pixels, y_pixels, 2, gdal.GDT_Float32)
    dataset.SetGeoTransform((x_min, PIXEL_SIZE, 0, y_max, 0, -PIXEL_SIZE))
    dataset.SetProjection(wkt_projection)
    dataset.GetRasterBand(1).WriteArray(array_1)
    dataset.GetRasterBand(2).WriteArray(array_2)
    dataset.FlushCache()


In [8]:
Magnitudes=[mag_1987,mag_1989,mag_1990,mag_1991,mag_1992,mag_1993,mag_1994,mag_1997,mag_1998,mag_1999,mag_2000,mag_2001,mag_2002,mag_2003,mag_2004]
Tiempos=[t_1987,t_1989,t_1990,t_1991,t_1992,t_1993,t_1994,t_1997,t_1998,t_1999,t_2000,t_2001,t_2002,t_2003,t_2004]
Nombres=['Inc_1987.tif','Inc_1989.tif','Inc_1990.tif','Inc_1991.tif','Inc_1992.tif','Inc_1993.tif','Inc_1994.tif','Inc_1997.tif','Inc_1998.tif','Inc_1999.tif','Inc_2000.tif','Inc_2001.tif','Inc_2002.tif','Inc_2003.tif','Inc_2004.tif']

In [9]:
# D:\giambelluca.144055\Incendios\Proyecto\Mag_Navarre Viejo

# C:\Users\giambelluca.144055\OneDrive\Incendios\Mag_Navarra
directory=input('Where do you want to save the result?: ')
os.chdir(directory)

for y in range(len(Magnitudes)):
    array_to_raster(Magnitudes[y],Tiempos[y],Nombres[y])

In [ ]:
Mag_Navarra_L7_cor

In [ ]:
# array_aleatorio = np.random.uniform(-1, 1, size=(7422, 7422)).astype(np.float32)

# array_con_nan = np.where(mag_1985 == 0, np.nan, mag_1985)

In [ ]:
# import gdal
# import matplotlib.pyplot as plt

# # Abre el dataset raster
# dataset = gdal.Open('./prueba.tiff')

# # Lee la primera banda del raster
# band = dataset.GetRasterBand(1)

# # Lee los datos de la banda como un array NumPy
# array = band.ReadAsArray()

# # Obtén la transformación geográfica del raster
# transform = dataset.GetGeoTransform()

# # Calcula las coordenadas geográficas de los píxeles
# x_origin = transform[0]
# y_origin = transform[3]
# pixel_width = transform[1]
# pixel_height = transform[5]

# # Crea una matriz de coordenadas x e y
# x_coords = x_origin + np.arange(array.shape[1]) * pixel_width
# y_coords = y_origin + np.arange(array.shape[0]) * pixel_height

# # Visualiza el array utilizando Matplotlib
# plt.imshow(array, cmap='viridis', extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])  
# plt.colorbar(label='Valor')
# plt.title('Visualización del raster con coordenadas geográficas')
# plt.xlabel('Longitud')
# plt.ylabel('Latitud')
# plt.show()



In [ ]:
# import rasterio
# from rasterio.windows import Window

# # Datos del array que deseas exportar
# data = mag_1986  # Reemplaza 'your_array' con tu array NumPy


# # Ejemplo
# export_path = 'mag_1986.tif'
# driver = 'GTiff'
# height, width = mag_1986.shape
# count = 1
# dtype = mag_1986.dtype
# crs = 'EPSG:4326'  # Define el CRS si lo conoces
# transform = None  # Define la transformación si la tienes

# # Abre el archivo para escritura
# with rasterio.open(
#     export_path, mode='w', driver=driver, height=height, width=width, count=count, dtype=dtype, crs=crs, transform=transform
# ) as dst:
#     # Define la ventana para escribir el array completo
#     # Escribe el array en el raster
#     dst.write(data.astype(rasterio.uint8), 1)

# # Cierra el archivo
# print(f'Raster exportado exitosamente: {export_path}')